In [6]:
from types import resolve_bases
import pickle
import numpy as np
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_log_error
import plotly.express as px
import xgboost as xgb
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from types import resolve_bases
import pickle

In [9]:
with open('/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/Modelling/ModelMk2.pkl', 'rb') as file:
    res = pickle.load(file)
X_train = res["X_train"]
X_test = res["X_test"]
y_train = res["y_train"]
y_test = res["y_test"]


In [10]:
tree_methods = ['approx', 'exact', 'hist']

space = {
'tree_method': hp.choice('tree_method', tree_methods),
'max_depth': hp.randint('max_depth', 3, 12),
'n_estimators': hp.randint('n_estimators', 1, 1000),
'num_parallel_tree': hp.randint('num_parallel_tree', 2, 8),
'min_child_weight': hp.randint('min_child_weight', 1, 200),
'subsample': hp.uniform('subsample', 0.7, 1),
'colsample_bytree': hp.uniform('colsample_bytree', 0.5, 1),
'colsample_bylevel': hp.uniform('colsample_bylevel', 0.5, 1),
'reg_lambda': hp.uniform('reg_lambda', 0, 10),
'learning_rate': hp.uniform('learning_rate', 0, 1),
}

def objective(params):
    xgb_model = xgb.XGBRegressor(**params)
    xgb_model.fit(X_train, y_train)
    y_pred = xgb_model.predict(X_test)
    RMSLE = np.sqrt(mean_squared_log_error(y_test, y_pred))
    return {'loss': RMSLE, 'status': STATUS_OK}

trials = Trials()

best_params = fmin(objective, space, algo=tpe.suggest, max_evals=100, trials=trials)
best_params["tree_method"] = tree_methods[best_params["tree_method"]]
print("Best set of hyperparameters: ", best_params)

100%|██████████| 100/100 [00:54<00:00,  1.82trial/s, best loss: 0.0654164036023744]
Best set of hyperparameters:  {'colsample_bylevel': np.float64(0.7147518312564775), 'colsample_bytree': np.float64(0.834057088768181), 'learning_rate': np.float64(0.5186269349701609), 'max_depth': np.int64(9), 'min_child_weight': np.int64(4), 'n_estimators': np.int64(17), 'num_parallel_tree': np.int64(6), 'reg_lambda': np.float64(2.414140826433988), 'subsample': np.float64(0.905145423380081), 'tree_method': 'hist'}


In [11]:
xgb_model = xgb.XGBRegressor(**{'colsample_bylevel': np.float64(0.6868508862097376), 'colsample_bytree': np.float64(0.7549621262553919), 'learning_rate': np.float64(0.15410629092509778), 'max_depth': np.int64(8), 'min_child_weight': np.int64(14), 'n_estimators': np.int64(862), 'num_parallel_tree': np.int64(6), 'reg_lambda': np.float64(9.76903393118995), 'subsample': np.float64(0.9143921578676975), 'tree_method': 'approx'})
xgb_model.fit(X_train, y_train)
y_pred = xgb_model.predict(X_test)
RMSLE = np.sqrt(mean_squared_log_error(y_test, y_pred))
print("The score is %.5f" % RMSLE )

The score is 0.06886


In [12]:
n = 10
iCoords_arr = np.linspace(0,1,n-1)
jCoords_arr = np.linspace(0,1,n-1)
kCoords_arr = np.linspace(0,1,n-1)
ijkCoords_lis = []
for i in iCoords_arr:
    for j in jCoords_arr:
        for k in kCoords_arr:
            ijkCoords_lis.append([i,j,k])
ijkCoords_arr = np.array(ijkCoords_lis)
y_pred = xgb_model.predict(ijkCoords_lis)
df2 = pd.DataFrame({'s1': ijkCoords_arr[:, 0],'s2': ijkCoords_arr[:, 1],'b1': ijkCoords_arr[:, 2], 'y_pred': y_pred})
df2["y_pred"] = y_pred
fig = px.scatter_3d(df2, x='s1', y='s2', z='b1', color='y_pred')
fig.show()
print(np.max(y_pred))
print(ijkCoords_arr[np.argmax(y_pred)])

16.49063
[0.    0.125 0.   ]


In [13]:
n = 200
iCoords_arr = np.linspace(0,1,n-1)
jCoords_arr = np.linspace(0,1,n-1)
kCoords_arr = np.linspace(0,1,n-1)
ijkCoords_lis = []
for i in iCoords_arr:
    for j in jCoords_arr:
        for k in kCoords_arr:
            ijkCoords_lis.append([i,j,k])
ijkCoords_arr = np.array(ijkCoords_lis)
y_pred = xgb_model.predict(ijkCoords_lis)
print(np.max(y_pred))
print(ijkCoords_arr[np.argmax(y_pred)])

17.265064
[0.33838384 0.64646465 0.77272727]
